## 第24章  优化算法：从 SGD 到 AdamW —— 训练加速的工程智慧

> 本章目标：从最基础的 SGD 出发，理解 Momentum（惯性）、Adam（自适应学习率）、AdamW（解耦 Weight Decay）三个关键升级的物理直觉和代码实现。掌握 Warmup + Cosine Decay 学习率调度，以及 Gradient Accumulation 攒批技巧。最后落地到 Agent 微调的两阶段学习率策略。
> 前置知识：第 12 章（梯度下降）、第 18 章（CLT）、第 23 章（梯度裁剪）


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print('env ready')


---
### 24.1  SGD 回顾 —— 纯梯度下降

第 12 章的核心公式：`θ -= lr * ∇L`。这就是 SGD（Stochastic Gradient Descent）——每一步沿负梯度方向迈一小步。简洁、直观、但有两个致命缺陷：

1. **峡谷震荡**：在狭长峡谷中（如 f(x,y)=x²+10y²），梯度指向谷底的方向和真正的最优方向存在偏移——SGD 沿峡谷壁反复震荡，收敛极慢
2. **学习率死板**：所有参数共享同一个 lr，但不同参数的梯度量级可能差 100 倍——对某些参数 lr 太大（震荡），对另一些参数 lr 太小（爬不动）

后续所有优化器，本质上都在解决这两个问题。


---
### 24.2  SGD + Momentum —— 给梯度加惯性

物理直觉：一个球从山上滚下来——它不仅受当前坡度（梯度）的影响，还带着之前积累的速度（动量）。即使走到一个局部平缓的地方，惯性会推着它继续向前，不会卡住。

数学翻译：引入速度变量 v，每一步更新 `v = β·v + (1−β)·∇L`，然后 `θ -= lr * v`。β≈0.9 是典型值——意味着当前更新中 90% 来自历史梯度方向，10% 来自当前梯度。

📐 **定义  Momentum**：`v = β·v + (1−β)·∇L`，`θ -= lr * v`。β 控制惯性大小：β=0 退化为 SGD，β 越接近 1 惯性越大。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x, y):
    return x**2 + 10 * y**2  # narrow valley -- SGD classic test function

def grad_f(x, y):
    return np.array([2*x, 20*y])

# Contour
X, Y = np.meshgrid(np.linspace(-3, 3, 100), np.linspace(-3, 3, 100))
Z = f(X, Y)

fig, ax = plt.subplots(figsize=(8, 7))
ax.contour(X, Y, Z, levels=30, cmap='viridis', alpha=0.5)

# SGD trajectory
x_sgd = np.array([2.5, 2.5])
lr = 0.05
path_sgd = [x_sgd.copy()]
for _ in range(50):
    x_sgd = x_sgd - lr * grad_f(*x_sgd)
    path_sgd.append(x_sgd.copy())
path_sgd = np.array(path_sgd)
ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'r-o', markersize=2, label='SGD (oscillates)')

# Momentum trajectory
x_mom = np.array([2.5, 2.5])
v = np.zeros(2)
beta = 0.9
path_mom = [x_mom.copy()]
for _ in range(50):
    v = beta * v + (1 - beta) * grad_f(*x_mom)
    x_mom = x_mom - lr * v
    path_mom.append(x_mom.copy())
path_mom = np.array(path_mom)
ax.plot(path_mom[:, 0], path_mom[:, 1], 'b-o', markersize=2, label='Momentum (smooth)')

ax.plot(0, 0, 'g*', markersize=15, label='Optimum (0,0)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_title('SGD vs Momentum: f=x^2+10y^2')
ax.legend(); ax.set_aspect('equal'); plt.show()
print('SGD: oscillates across valley walls, slow convergence')
print('Momentum: inertia suppresses oscillation, accelerates along valley floor')


> **关键洞察**：Momentum 在峡谷函数上的轨迹比 SGD 平滑得多——它不会被峡谷壁弹来弹去，而是像有记忆一样沿着谷底的总体方向加速前进。这就是为什么 Momentum 几乎在所有深度学习任务中都优于纯 SGD。


---
### 24.3-24.4  Adam → AdamW —— 自适应学习率 + 解耦 Weight Decay

Adam 是 SGD+Momentum 的两个关键升级：

1. **动量项（一阶矩 m）**：和 Momentum 的 v 一样——积累历史梯度方向，平滑更新
2. **自适应项（二阶矩 v）**：累积历史梯度**平方**——梯度方差大的参数自动缩小步长，方差小的参数自动放大步长

📐 **定义  Adam**：m = β₁m + (1-β₁)g,  v = β₂v + (1-β₂)g²。θ -= lr * m_hat / (sqrt(v_hat) + ε)。默认参数：lr=1e-3, β₁=0.9, β₂=0.999, ε=1e-8。

**AdamW 的关键修复**：把 Weight Decay 从梯度中解耦出来——θ -= lr * (m_hat / (sqrt(v_hat) + ε) + λ * θ)。HuggingFace Trainer 默认用 AdamW。

**Adam 是训练 Transformer 的事实标准**——不理解它就无法 debug 训练曲线。


In [ ]:
import numpy as np

def f(x, y): return x**2 + 10 * y**2
def grad_f(x, y): return np.array([2*x, 20*y])

def optimize(opt_name, lr=0.05, n_steps=200):
    x = np.array([2.5, 2.5])
    path = [x.copy()]

    if opt_name == 'Adam':
        m, v2 = np.zeros(2), np.zeros(2)
        beta1, beta2, eps = 0.9, 0.999, 1e-8
        for t in range(1, n_steps+1):
            g = grad_f(*x)
            m = beta1*m + (1-beta1)*g
            v2 = beta2*v2 + (1-beta2)*g**2
            m_hat = m/(1-beta1**t); v_hat = v2/(1-beta2**t)
            x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
            path.append(x.copy())

    elif opt_name == 'Momentum':
        v = np.zeros(2); beta = 0.9
        for _ in range(n_steps):
            v = beta*v + (1-beta)*grad_f(*x)
            x = x - lr * v
            path.append(x.copy())

    elif opt_name == 'SGD':
        for _ in range(n_steps):
            x = x - lr * grad_f(*x)
            path.append(x.copy())

    return np.array(path)

for name in ['SGD', 'Momentum', 'Adam']:
    path = optimize(name)
    final_dist = np.linalg.norm(path[-1])
    print(f'{name:10s}: final distance to optimum = {final_dist:.6f}')


> **关键洞察**：Adam 用二阶矩 v 为每个参数独立调整学习率——在峡谷中，y 方向的梯度（20y）远大于 x 方向（2x），Adam 自动给 y 方向更小的有效步长、x 方向更大的有效步长，完美适应各向异性的损失曲面。这就是为什么 Adam 不需要精细调参就能在大多数任务上表现良好。


---
### 24.5  学习率调度 —— Warmup + Cosine Decay

Transformer 训练中，前几千步需要**极小的学习率**——因为训练初期梯度方向不稳定，大步更新可能直接炸穿。Warmup 策略：前 4000 步 lr 从 0 线性增长到目标值，之后再按 Cosine 曲线衰减到 0。

📐 **定义  Warmup + Cosine Decay**：warmup 阶段 lr 线性从 0 增长到 peak_lr；之后 lr = 0.5 * peak_lr * (1 + cos(π*(t−warmup)/(total−warmup)))。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

total_steps = 10000
warmup_steps = 2000
peak_lr = 1e-3

t = np.arange(1, total_steps + 1)
# Constant
lr_const = np.full(total_steps, peak_lr)
# Step decay
lr_step = peak_lr * 0.5 ** (t // 3000)
# Cosine with warmup
warmup = np.minimum(t / warmup_steps, 1.0)
cosine = 0.5 * (1 + np.cos(np.pi * (t - warmup_steps) / (total_steps - warmup_steps)))
lr_cosine = peak_lr * np.where(t <= warmup_steps, warmup, cosine)

plt.figure(figsize=(10, 4))
plt.plot(t, lr_const, label='Constant')
plt.plot(t, lr_step, label='Step Decay')
plt.plot(t, lr_cosine, label='Warmup + Cosine', linewidth=2)
plt.axvline(x=warmup_steps, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Step'); plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedules'); plt.legend(); plt.grid(alpha=0.3); plt.show()


---
### 24.6-24.7  Gradient Accumulation + Agent 微调学习率策略

**Gradient Accumulation**：每 N 次 loss.backward() 后调用一次 optimizer.step()，等效 batch_size 放大 N 倍。解决显存不够又想用大 batch 的矛盾。

**Agent 微调两阶段 LR**：
- **指令微调**阶段用 Warmup + Cosine Decay（标准配置，lr=2e-5）
- **RLHF 阶段**用极低学习率（1e-6）防止灾难性遗忘

💡 实践中：第一阶段 lr=2e-5，第二阶段 lr=1e-6，两者差 20 倍。如果用恒定高学习率做 RLHF，模型会迅速过拟合 reward 信号、丧失通用语言能力。

🔗 **AI 连接**：第 31 章训练循环中 optimizer.step() 调用的是 AdamW。第 34 章微调 GPT-2 时会实际对比不同学习率策略的 loss 曲线。


---
**✏️ 习题**

> 在下方新建代码单元格作答。

1. （概念）Momentum 的 β=0.9 意味着什么？β=0 退化成什么？β=1 会怎样？

2. （概念）Adam 的自适应学习率指的是什么？为什么对稀疏特征特别有效？

3. （代码）在 f(x,y)=x²+10y² 上对比 SGD、Momentum、Adam 的收敛路径，画二维等高线+三条轨迹图。

4. （代码）实现 Warmup + Cosine Decay 调度器，画学习率曲线。验证 warmup 阶段 lr 单调递增、decay 阶段单调递减。


---

> 🔗 **章末钩子**：优化器决定了怎么走。但起点在哪同样关键——全零初始化的对称性陷阱、Xavier/Kaiming 的方差保持原理。
> 预览：下一章——**权重初始化**，起点决定终点。

> 💡 **提示**：完成后运行 Kernel -> Restart & Run All 验证所有代码块。
